# Does a random forest get worse when we add more trees

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

In a previous notebook we saw that an AdaBoost becomes worse as we add trees when the target has a long tail, and that the same never happens in classification.

Here we ask the same question of the random forest, with the same datasets and the same experiment, and at the end we put the two models side by side.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import (
    fetch_california_housing,
    fetch_openml,
    load_diabetes,
    make_classification,
)
from sklearn.ensemble import (
    AdaBoostRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.metrics import roc_auc_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore")

### The experiment

AdaBoost gave us the performance after each tree with `staged_predict`, and the random forest has nothing of the sort, because its trees are independent and the order in which we add them is arbitrary.

We obtain the same curve in another way. A random forest predicts the average of its trees, so we ask every tree for its prediction, and then the average of the first tree, of the first two trees, of the first three, and so on, is the forest that we would have obtained with that many trees.

In [ ]:
def prepare(X, y, max_rows=20000):
    X = pd.DataFrame(X).copy()
    for var in X.columns:
        if str(X[var].dtype) in ("object", "category", "str"):
            X[var] = X[var].astype("category").cat.codes
    X = X.astype(float).fillna(-1)
    y = pd.Series(np.asarray(y))
    # we work with a sample of the largest datasets, to keep the notebook fast
    if len(X) > max_rows:
        X, _, y, _ = train_test_split(X, y, train_size=max_rows, random_state=0)
    return X, y


def running_average(predictions):
    trees = np.arange(1, len(predictions) + 1)
    return np.cumsum(predictions, axis=0) / trees[:, None]

In [ ]:
def classification_curve(X, y, n_estimators=500):
    X, y = prepare(X, y)
    y = y.astype("category").cat.codes
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=0
    )
    forest = RandomForestClassifier(n_estimators=n_estimators, random_state=0, n_jobs=-1)
    forest.fit(X_train, y_train)

    predictions = np.array(
        [tree.predict_proba(X_test.to_numpy())[:, 1] for tree in forest.estimators_]
    )
    return pd.Series(
        [roc_auc_score(y_test, p) for p in running_average(predictions)],
        index=range(1, n_estimators + 1),
    )


def regression_curve(X, y, n_estimators=300):
    X, y = prepare(X, y)
    y = y.astype(float)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    forest = RandomForestRegressor(n_estimators=n_estimators, random_state=0, n_jobs=-1)
    forest.fit(X_train, y_train)

    predictions = np.array([tree.predict(X_test.to_numpy()) for tree in forest.estimators_])
    return pd.Series(
        [root_mean_squared_error(y_test, p) for p in running_average(predictions)],
        index=range(1, n_estimators + 1),
    )

In [ ]:
def summarise(curves, higher_is_better):
    summary = {}
    for name, curve in curves.items():
        best = curve.max() if higher_is_better else curve.min()
        summary[name] = {
            "best": best,
            "best at tree": curve.idxmax() if higher_is_better else curve.idxmin(),
            "final": curve.iloc[-1],
            "lost from best %": 100 * abs(curve.iloc[-1] - best) / best,
        }
    return pd.DataFrame(summary).T.round(4)

### Classification

We start with data that we make ourselves, with clean labels and with the labels of ten and twenty percent of the observations flipped, and we train five hundred trees.

In [ ]:
X, y = make_classification(n_samples=6000, n_features=20, n_informative=6, random_state=0)

curves = {"clean labels": classification_curve(X, y)}
for noise in [0.1, 0.2]:
    rng = np.random.default_rng(0)
    noisy = np.where(rng.random(len(y)) < noise, 1 - y, y)
    curves[f"{int(noise * 100)}% wrong labels"] = classification_curve(X, noisy)

synthetic_curves = pd.DataFrame(curves)
summarise(curves, higher_is_better=True)

In [ ]:
synthetic_curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("A random forest with data that we made ourselves")
plt.show()

In [ ]:
datasets = {
    "credit-g": 31,
    "diabetes": 37,
    "spambase": 44,
    "phoneme": 1489,
    "churn": 40701,
}

curves = {}
for name, data_id in datasets.items():
    data = fetch_openml(data_id=data_id, as_frame=True)
    curves[name] = classification_curve(data.data, data.target)

classification_curves = pd.DataFrame(curves)
summarise(curves, higher_is_better=True)

In [ ]:
classification_curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("A random forest in five classification datasets")
plt.legend(loc="lower right")
plt.show()

The curves climb steeply with the first trees, they flatten, and they stay there. This is what we also saw with AdaBoost, so in classification both models let us add trees without fear.

### Regression

These are the six datasets that broke AdaBoost, three of them with a long right tail and three of them with a tame target.

In [ ]:
rent = pd.read_csv("../prepare-data/apartment_rent.csv", index_col="time")
california = fetch_california_housing(as_frame=True)
house_sales = fetch_openml(data_id=42731, as_frame=True)
ames = fetch_openml(data_id=42165, as_frame=True)
abalone = fetch_openml(data_id=44956, as_frame=True)
diabetes = load_diabetes(as_frame=True)

datasets = {
    "apartment rent": (rent.drop(columns="price"), rent["price"]),
    "house sales": (house_sales.data, house_sales.target),
    "california housing": (california.data, california.target),
    "ames housing": (ames.data, ames.target),
    "abalone": (abalone.data, abalone.target),
    "diabetes": (diabetes.data, diabetes.target),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, (name, (X, y)) in zip(axes.flat, datasets.items()):
    target = pd.Series(np.asarray(y, dtype=float))
    ax.hist(target, bins=50)
    ax.set_title(f"{name}, skew {target.skew():.1f}")
fig.suptitle("The targets")
fig.tight_layout()
plt.show()

In [ ]:
curves = {name: regression_curve(X, y) for name, (X, y) in datasets.items()}

summary = summarise(curves, higher_is_better=False)
summary["skew of the target"] = [
    pd.Series(np.asarray(y, dtype=float)).skew() for _, y in datasets.values()
]
summary.sort_values("skew of the target", ascending=False).round(2)

In [ ]:
# the targets are in different units, so we divide each curve by its own best value
relative = pd.DataFrame({name: curve / curve.min() for name, curve in curves.items()})

relative.plot()
plt.xlabel("Number of trees")
plt.ylabel("rmse divided by the best rmse")
plt.title("A random forest in six regression datasets")
plt.show()

Every curve falls and then stays flat, including the rent and the house sales, which are the two targets with the longest tail. The forest loses less than one percent from its best point in all of them, and that difference is noise.

The shape of the target, which decided everything for AdaBoost, does not matter here.

### The two models side by side

We now train an AdaBoost and a random forest in the three datasets with the long tail, and we plot the two curves of each dataset together.

In [ ]:
def adaboost_curve(X, y, n_estimators=100):
    X, y = prepare(X, y)
    y = y.astype(float)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=3), n_estimators=n_estimators, random_state=0
    )
    model.fit(X_train, y_train)
    return pd.Series(
        [root_mean_squared_error(y_test, p) for p in model.staged_predict(X_test)],
        index=range(1, len(model.estimators_) + 1),
    )


skewed = ["apartment rent", "house sales", "california housing"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, name in zip(axes, skewed):
    features, target = datasets[name]
    forest = curves[name]
    boosting = adaboost_curve(features, target)
    ax.plot(forest.index, forest / forest.min(), label="random forest")
    ax.plot(boosting.index, boosting / boosting.min(), label="adaboost")
    ax.set_xlabel("Number of trees")
    ax.set_ylabel("rmse divided by the best rmse")
    ax.set_title(name)
    ax.legend()
plt.tight_layout()
plt.show()

The reason is in how the two models build the ensemble.

The random forest trains every tree on its own, on a different sample of the data and with a different sample of the features, and no tree knows anything about the trees that came before it. The prediction is their average, and averaging more numbers that are wrong in different directions can only help, so the curve falls and then flattens.

AdaBoost trains every tree on the weights that the previous trees left behind. When the target has a long tail, a handful of extreme observations take the weights, every new tree chases them, and the ensemble walks away from the rest of the data.

### What we take home

The number of trees of a random forest is not a hyperparameter that we need to tune. More trees never made the model worse in any of the eleven datasets that we tried, with clean labels and with labels that we spoiled, with tame targets and with targets with a very long tail, so we can set a large number and spend our time tuning the depth of the trees and the number of features instead.

With AdaBoost the number of trees is a real hyperparameter, because the trees are not independent and the mistakes of one round decide what the next round sees.

This is the practical difference between averaging and boosting, and it is worth remembering when we choose which of the two to train.